# MuSiQue ở n=500 - sửa chỗ bài tự nhận là yếu nhất

Mục Limitations của cả hai bản viết thẳng:

> *MuSiQue stays at $n=50$, so every English control, including the one behind
> our granularity-not-typology claim, rests on the loosest sample in the paper.*

Khẳng định *"hư hại ngữ pháp là chuyện **granularity của bộ nén**, không phải
**loại hình ngôn ngữ**"* dựa vào đối chứng tiếng Anh. Đối chứng đó đang ở
**n=50**, ngưỡng phát hiện ghép cặp **~18 F1** - tức gần như không kết luận
được gì.

Ở **n=500** ngưỡng xuống khoảng **5 F1**, ngang cấp với VimQA (đã có n=1003).

| | |
|---|---|
| Chi phí | ~1.8h T4 (MuSiQue dài hơn VimQA 1.38×: 1485 vs 1080 token/câu) |
| Phương pháp | `raw,oracle,llmlingua2,itercomp` - bốn cái của bảng chính |
| $k$ | 90, đúng như paper §5.1 cho MuSiQue |

**Nếu kết quả đổi thì phải sửa bài.** Đó là lý do nên chạy, không phải lý do
tránh.

In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn bằng cách clone repo. Trước đây repo còn private nên notebook
# phải tải qua một gist công khai; repo public rồi, nên clone thẳng là nguồn duy
# nhất, không còn hai bản phải giữ đồng bộ.
import os, sys, subprocess, shutil

BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'ANLP-IterCOMP')
GIT_URL = 'https://github.com/nhantrnh/ANLP-IterCOMP.git'
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data', 'metrics', 'fertility', 'stats', 'budget']

os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]

subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO], check=True)

miss = [m for m in NEED
        if not os.path.exists(os.path.join(REPO, 'src', 'itercomp', m + '.py'))]
if miss:
    raise SystemExit('Bản clone thiếu module: ' + ', '.join(miss))

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

sha = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                     capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| src/scripts/:', os.path.isdir('src/scripts'))
print('đủ', len(NEED), 'module | commit:', sha)


In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ MuSiQue n=500, reader 7B ══ ~1.8h tren T4
# run_eval.py ghi checkpoint sau MOI cau, nen het gio van chay tiep duoc.
import subprocess, sys, os, time
READER='Qwen/Qwen2.5-7B-Instruct'; OUT='results/musique_500_7b.json'
os.makedirs('results', exist_ok=True)

def run_stream(cmd, logfile):
    t0=time.time()
    with open(logfile,'w') as lf:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

CMD=[sys.executable,'-u','scripts/run_eval.py',
     '--dataset','musique','--limit','500',
     '--reader','hf','--reader-model',READER,'--load-4bit',
     '--itercomp-llm','hf','--scorer','dual',
     '--methods','raw,oracle,llmlingua2,itercomp','--out',OUT]

if os.path.exists(OUT):
    print(f'[bo qua] da co {OUT}')
else:
    rc=run_stream(CMD,'results/log_musique500.txt')
    assert rc==0, f'that bai (rc={rc})'

In [ ]:
# So voi MuSiQue n=50 da co: thu hang co giu khong?
import json
new=json.load(open('results/musique_500_7b.json'))['summary']
print(f"{'method':14s} {'F1* n=500':>10s} {'giu%':>7s}")
for m,v in sorted(new.items(), key=lambda kv:-kv[1]['f1_norm']):
    print(f"{m:14s} {v['f1_norm']:10.2f} {100*v['ratio']:6.1f}%")
print('\nn=50 (da co):  oracle 57.05 > itercomp 35.11 > raw 32.85 > llmlingua2 13.51')

In [ ]:
import shutil, os
BASE='/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z=shutil.make_archive(os.path.join(BASE,'results_musique500'),'zip','results')
print('✓',z,f'({os.path.getsize(z)/1e6:.1f} MB)')